# Modeling and tuning — Step 11

**CSE437 Data Science | Group 15 | Owner: Sadat | Next: Step 12**

This notebook implements the majority baseline plus logistic-regression and random-forest families. Each learned family compares all Step 9 features with Step 10's selected-feature rule. Model hyperparameter search remains Step 12; final refit/test evaluation remains Step 13.

The approved problem, questions, dataset, target, cohort and frozen split remain unchanged. The final test is not transformed or scored. Training prefixes fit all preprocessing and feature selection; validation labels are used only to measure performance.

**Execution provenance:** the six code cells below executed sequentially in a fresh Python process with actual stdout saved. This runtime lacks Jupyter/IPython/nbformat; a full fresh Jupyter-kernel run and canonical format validation remain final submission gates. No model scores are illustrative.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
ROOT=Path.cwd().resolve()
if not (ROOT/"src").is_dir() and (ROOT.parent/"src").is_dir():
    ROOT=ROOT.parent
assert (ROOT/"data/splits/step6_split_plan.json").is_file()
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.model_comparison import PROTOCOL,run_model_comparison
from src.modeling import make_model_pipeline
print(json.dumps(PROTOCOL,indent=2))


{
  "candidates": [
    {
      "candidate": "majority",
      "family": "majority",
      "representation": "none"
    },
    {
      "candidate": "lr_full",
      "family": "logistic_regression",
      "representation": "full"
    },
    {
      "candidate": "lr_selected",
      "family": "logistic_regression",
      "representation": "selected"
    },
    {
      "candidate": "rf_full",
      "family": "random_forest",
      "representation": "full"
    },
    {
      "candidate": "rf_selected",
      "family": "random_forest",
      "representation": "selected"
    }
  ],
  "model_settings": {
    "logistic_regression": {
      "C": 1.0,
      "solver": "lbfgs",
      "max_iter": 2000,
      "tol": 0.0001,
      "class_weight": null,
      "random_state": 42
    },
    "random_forest": {
      "n_estimators": 100,
      "criterion": "gini",
      "max_depth": null,
      "min_samples_split": 2,
      "min_samples_leaf": 1,
      "max_features": "sqrt",
      "bootstrap": true,
    

## Models and validation

| Candidate | Purpose / fixed setup |
| --- | --- |
| Majority baseline | Predict the majority class observed in each training fold; ignores features |
| Logistic regression, full / selected | Linear regularized reference; C=1, lbfgs, max_iter=2000, tol=1e−4; scaled numeric inputs |
| Random forest, full / selected | Nonlinear tree interactions; 100 trees, unlimited depth, leaf size 1, sqrt feature sampling, bootstrap, no class weighting; unscaled numeric inputs |

Seeds are 42. Threshold is 0.5 (probability ≥0.5 predicts cancellation). No class resampling, threshold optimization or model-parameter search occurs. Three expanding development folds provide the same rows to each candidate. Every model gets a fresh pipeline; feature selection is fitted on training labels only. Mean cancellation F1 across the three folds is primary; accuracy, precision, recall and ROC-AUC are secondary.

The first run writes the comparison protocol before fitting and rejects unversioned protocol changes. Logistic metrics are checked against the published Step 10 reference. The random-forest setup is an untuned starting point; its depth and leaf size may permit overfitting, which the diagnostics below expose.


In [2]:
summary,comparison,fold_results=run_model_comparison(ROOT)
print("\nPreferred untuned candidate:",summary["preferred_candidate"])
print("Final test rows processed:",summary["test_rows_fitted_transformed_or_scored"])


Step 11: fitting majority, fold 1 (23,797 training rows)
Completed majority, fold 1: validation F1=0.000000; training F1=0.000000
Step 11: fitting lr_full, fold 1 (23,797 training rows)
Completed lr_full, fold 1: validation F1=0.643231; training F1=0.854044
Step 11: fitting lr_selected, fold 1 (23,797 training rows)
Completed lr_selected, fold 1: validation F1=0.693691; training F1=0.850240
Step 11: fitting rf_full, fold 1 (23,797 training rows)
Completed rf_full, fold 1: validation F1=0.657928; training F1=0.987615
Step 11: fitting rf_selected, fold 1 (23,797 training rows)
Completed rf_selected, fold 1: validation F1=0.667216; training F1=0.987439
Step 11: fitting majority, fold 2 (47,690 training rows)
Completed majority, fold 2: validation F1=0.000000; training F1=0.000000
Step 11: fitting lr_full, fold 2 (47,690 training rows)
Completed lr_full, fold 2: validation F1=0.702455; training F1=0.804477
Step 11: fitting lr_selected, fold 2 (47,690 training rows)
Completed lr_selected, f

## Development comparison

Scores describe the fixed development windows, not final test performance. The folds are reused for model/representation choice, so the winning mean can be optimistic. Across-fold SD is descriptive, not a confidence interval.

![Untuned model comparison](../figures/07_model_comparison.png)


In [3]:
print(comparison[["candidate","mean_f1","fold_sd_f1","mean_accuracy","mean_precision","mean_recall","mean_roc_auc"]].round(6).to_string(index=False))
print("\nF1 by forward validation period:")
print(fold_results.pivot(index="fold",columns="candidate",values="f1").round(6).to_string())


  candidate  mean_f1  fold_sd_f1  mean_accuracy  mean_precision  mean_recall  mean_roc_auc
   majority 0.000000    0.000000       0.638167        0.000000     0.000000      0.500000
    lr_full 0.693094    0.045904       0.751370        0.681025     0.753386      0.876098
lr_selected 0.713609    0.019669       0.809314        0.783364     0.659498      0.882905
    rf_full 0.626504    0.047179       0.793498        0.906662     0.481119      0.886473
rf_selected 0.657994    0.016997       0.803866        0.893717     0.521714      0.892267

F1 by forward validation period:
candidate   lr_full  lr_selected  majority   rf_full  rf_selected
fold                                                             
1          0.643231     0.693691       0.0  0.657928     0.667216
2          0.702455     0.714117       0.0  0.572253     0.638378
3          0.733595     0.733020       0.0  0.649330     0.668386


## Imbalance and training-versus-validation diagnostics

A majority baseline can have substantial accuracy while missing every cancellation. This explains why cancellation F1, recall and precision matter. Training scores are in-sample resubstitution diagnostics; gaps can reflect both overfitting and temporal shift, with repeated profiles adding another limitation. They are not independent generalization estimates.


In [4]:
print(comparison[["candidate","mean_training_f1","mean_f1","mean_f1_gap","mean_fit_seconds"]].round(6).to_string(index=False))
balance=fold_results.query("candidate=='majority'")[["fold","train_rows","training_canceled","validation_rows","validation_canceled","training_majority_class"]].copy()
balance["training_cancellation_percent"]=100*balance.training_canceled/balance.train_rows
balance["validation_cancellation_percent"]=100*balance.validation_canceled/balance.validation_rows
print("\nFold class composition (development only):")
print(balance.round(3).to_string(index=False))
print("\nAggregate confusion counts are saved per candidate/fold in fold_results.csv.")


  candidate  mean_training_f1  mean_f1  mean_f1_gap  mean_fit_seconds
   majority          0.000000 0.000000     0.000000          0.002263
    lr_full          0.817267 0.693094     0.124173          1.494870
lr_selected          0.814529 0.713609     0.100920          1.370380
    rf_full          0.990300 0.626504     0.363796         20.098629
rf_selected          0.990202 0.657994     0.332209         19.409045

Fold class composition (development only):
 fold  train_rows  training_canceled  validation_rows  validation_canceled  training_majority_class  training_cancellation_percent  validation_cancellation_percent
    1       23797               8561            23893                 8573                        0                         35.975                           35.881
    2       47690              17134            23776                 8867                        0                         35.928                           37.294
    3       71466              26001        

## Pipeline and reproduction checks

The saved evidence includes full estimator parameters, each fold's feature names, confusion counts, membership hashes and frozen input checksums. Prediction must not change the learned representation. Logistic results must match Step 10's full/selected reference. No fitted global model or row-level predictions are exported at this stage.


In [5]:
assert summary["model_fits"]==15 and summary["candidate_count"]==5
assert summary["logistic_results_match_step10"]
assert fold_results.representation_unchanged_after_prediction.all()
assert not fold_results.convergence_warning.any()
assert (fold_results[["tn","fp","fn","tp"]].sum(axis=1)==fold_results.validation_rows).all()
assert fold_results.groupby("fold").validation_membership_sha256.nunique().eq(1).all()
print("15 fits verified: identical validation membership within each fold and unchanged representation state.")
print("\nEncoded output widths:")
print(fold_results.pivot(index="fold",columns="candidate",values="encoded_columns").to_string())
print("\nLogistic metrics reproduce Step 10; frozen source/split checksums verified.")


15 fits verified: identical validation membership within each fold and unchanged representation state.

Encoded output widths:
candidate  lr_full  lr_selected  majority  rf_full  rf_selected
fold                                                           
1              332          247         0      332          247
2              421          314         0      421          314
3              490          366         0      490          366

Logistic metrics reproduce Step 10; frozen source/split checksums verified.


## Handoff — Step 12, Sadat

Use the leading untuned candidate as a starting point, not a final answer. Step 12 must report model search spaces, search method, candidate/fold counts and results for the two learned families. Keep the same development folds, primary metric and threshold policy. Representation preferences may differ by family; retain evidence for those choices.

Keep every learned preprocessing/selection step inside the pipeline passed to the search. Do not use a globally preprocessed matrix, validation labels to fit selectors, or held-out scores to choose settings. Freeze all choices before Step 13's final refit/test evaluation.


In [6]:
next_pipeline=make_model_pipeline(summary["preferred_family"],summary["preferred_representation"])
assert not hasattr(next_pipeline.named_steps.get("representation"),"preprocessor_")
assert summary["test_rows_fitted_transformed_or_scored"]==0
assert summary["model_hyperparameter_tuning_completed"] is False
print("Step 11 complete. Current untuned leader:",summary["preferred_candidate"])
print("Step 12 — Sadat: train-fold model hyperparameter search; test still reserved for Step 13.")


Step 11 complete. Current untuned leader: lr_selected
Step 12 — Sadat: train-fold model hyperparameter search; test still reserved for Step 13.
